# DeepSun/FlareML independent data reconstruction (2026)

This notebook uses **new post-MSc code**, not the lost original 2024 notebook. The source dataset remains attributed to the DeepSun/FlareML authors. See `README.md` for provenance and limitations.

## Preparation

First install dependencies from `requirements.txt`, then run `python fetch_upstream.py` from the reconstruction directory. The script verifies the exact upstream Git blob before saving the local CSV.

In [ ]:
from pathlib import Path
import pandas as pd
from experiment import load_data, evaluate, FEATURES

source = Path('local_data/flaringar_original_data.csv')
df = load_data(source)
print('Rows:', len(df), 'Unique active regions:', df['AR'].nunique())
display(df['target'].value_counts().reindex(['B','C','M','X']))

## New baseline comparison

The baseline uses fold-safe scaling. These numbers, when executed, are **new 2026 results** and must not be presented as reproduced 2024 thesis figures.

In [ ]:
result = evaluate(df, mode='stratified', model='extra_trees', sampling='none', folds=5)
pd.DataFrame(result['folds'])[['fold','train_size','test_size','balanced_accuracy','tss_ovr_macro']]

## Validation alternatives

Explore `mode='grouped'` (active-region IDs) and `mode='chronological'` (80/20 time holdout), and `sampling='under'` or `'smote'`. Each setting is an independently defined new methodological experiment, not a claim about the submitted analysis.

## Data balance and validation sensitivity

The charts below use the verified public upstream CSV and separately evaluated 2026 runs. Every metric comparison here uses the thesis-defined mean of four one-vs-rest BACC values. The 80/20 temporal split has no X-class test cases, so it is intentionally excluded from the four-class comparison.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

counts = df.target.value_counts().reindex(['B','C','M','X'])
fig, ax = plt.subplots(figsize=(7.4, 3.5))
counts.plot(kind='bar', ax=ax)
ax.set_title('DeepSun reference data: class imbalance (845 events)')
ax.set_xlabel('Solar flare class')
ax.set_ylabel('Number of events')
ax.tick_params(axis='x', labelrotation=0)
for i, val in enumerate(counts):
    ax.text(i, val + 5, str(int(val)), ha='center')
fig.tight_layout()
plt.show()

In [ ]:
results = pd.read_csv('local_results/matrix_summary.csv')
pairs = results[results['mode'].isin(['stratified', 'grouped'])].copy()
pivot = pairs.pivot_table(index=['model', 'sampling'], columns='mode',
                          values='mean_thesis_style_ovr_bacc').sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
pivot[['stratified', 'grouped']].plot.bar(ax=ax)
ax.set_title('2026 baseline sensitivity to active-region grouping')
ax.set_ylabel('Macro one-vs-rest BACC (thesis metric definition)')
ax.set_xlabel('Model and training-fold sampling')
ax.set_ylim(0, 0.85)
ax.legend(title='Validation', loc='upper center', bbox_to_anchor=(0.5, 1.24), ncol=2, frameon=False)
ax.tick_params(axis='x', labelrotation=30)
fig.tight_layout()
plt.show()

## Scientific interpretation

The grouped folds prevent identical active-region IDs from appearing in training and evaluation. Repeated active regions all share the same class label in this dataset. Ordinary stratified folds do not respect that dependence and generally report higher scores. This does **not** establish the size of the bias in the 2024 dissertation, whose precise code, environment and experimental protocol have not been recovered. The chronological 2014+ sensitivity experiment and seed-variation results are recorded in the separate CSV audit artefacts.